# 🎯 Modelo Preditivo Balanceado com SMOTE

Este notebook tem como objetivo construir um modelo de classificação utilizando técnicas de balanceamento de classes com SMOTE, padronização dos dados e validação cruzada para evitar overfitting.

- Dados: `leads_processados_para_pbi.csv`
- Target: `convertido`
- Algoritmo principal: RandomForestClassifier

In [4]:
# 📦 Importação de bibliotecas
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE

In [13]:
# 📥 Leitura dos dados
df = pd.read_csv('../../Data/PROCESSED/leads_processados_para_pbi.csv')
df.head()

,lead_id,origem,data_cadastro,status_conversao,dias_ate_1o_trade,valor_deposito,perfil,pais,foi_convertido,ano_mes_cadastro
0,210428,Instagram,2025-01-07,Não Convertido,NaN,0.0,Moderado,Ukraine,False,2025-01
1,210429,E-mail Marketing,2025-01-08,Não Convertido,NaN,0.0,Agressivo,Burundi,False,2025-01
2,210430,Instagram,2025-01-07,Não Convertido,NaN,0.0,Moderado,Korea,False,2025-01
3,210431,Orgânico,2025-01-10,Não Convertido,NaN,0.0,Conservador,Turks and Caicos Islands,False,2025-01
4,210432,YouTube Ads,2025-01-11,Não Convertido,NaN,0.0,Conservador,Luxembourg,False,2025-01


In [14]:
# 🎯 Transformando a variável target em binária
df['target'] = df['status_conversao'].map({'Convertido': 1, 'Não Convertido': 0})


In [15]:
# 🧹 Removendo colunas não úteis
df.drop(columns=['status_conversao', 'lead_id', 'data_cadastro'], inplace=True)

In [16]:
# 📊 Separando X e y
X = df.drop(columns=['target'])
y = df['target']

In [17]:
# 🔄 Convertendo variáveis categóricas (origem, país, perfil, etc.)
X = pd.get_dummies(X, drop_first=True)

In [10]:
# 📏 Padronização dos dados
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [11]:
# 📂 Separando treino e teste
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, stratify=y, random_state=42)


In [12]:
# ✅ Pronto para modelagem!
print("Shape dos dados de treino:", X_train.shape)
print("Proporção de classes no target:\n", y.value_counts(normalize=True))

Shape dos dados de treino: (213958, 280)
Proporção de classes no target:
 target
0    0.949643
1    0.050357
Name: proportion, dtype: float64


In [ ]:
# 🎯 Definindo target e preditores
y = df['status_conversao']
X = df.drop(columns=['data_cadastro', 'ano_mes_cadastro'], errors='ignore')
# Converte variáveis categóricas para dummies
X = pd.get_dummies(X, columns=['origem'], drop_first=True)

In [4]:

# 🧹 Tratamento de valores ausentes com lógica de negócio
# Se o lead não foi convertido, assumimos que ele nunca fez trade (0 dias até o 1º trade)
X['dias_ate_1o_trade'] = X['dias_ate_1o_trade'].fillna(0)


In [5]:
# ⚠️ Verifica colunas categóricas restantes
cat_cols = X.select_dtypes(include='object').columns.tolist()
print("Colunas categóricas encontradas:", cat_cols)

# ✅ Converte todas as categóricas para variáveis dummies
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)


Colunas categóricas encontradas: ['status_conversao', 'perfil', 'pais']


In [6]:
# 📊 Padronização dos dados
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [7]:
# Verifica colunas com valores ausentes
print(X.isna().sum())

lead_id                   0
dias_ate_1o_trade         0
valor_deposito            0
foi_convertido            0
origem_Facebook           0
                         ..
pais_Wallis and Futuna    0
pais_Western Sahara       0
pais_Yemen                0
pais_Zambia               0
pais_Zimbabwe             0
Length: 258, dtype: int64


In [8]:
# ⚖️ Aplicando SMOTE para balanceamento
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_scaled, y)

print("Distribuição após SMOTE:")
print(pd.Series(y_resampled).value_counts(normalize=True))

c:\Users\leojo\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] O sistema não pode encontrar o arquivo especificado
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\leojo\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\leojo\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\leojo\AppData\Local\Programs\Python\Python313\Lib\subprocess.py",

Distribuição após SMOTE:
status_conversao
Não Convertido    0.5
Convertido        0.5
Name: proportion, dtype: float64


In [9]:
# 🔁 Validação cruzada com RandomForest
model = RandomForestClassifier(n_estimators=100, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X_resampled, y_resampled, cv=cv, scoring='roc_auc')

print(f"AUC Média (RandomForest): {scores.mean():.4f}")

AUC Média (RandomForest): 1.0000
